# TracIn Implementation: California Housing Prices (Regression)

This notebook reproduces the regression experiment from Section 5.1 of the paper *Estimating Training Data Influence by Tracing Gradient Descent*. 

The goal of this task is to identify "comparables"—training examples that had the most influence (proponents and opponents) on the model's prediction for a specific test house. 

**Paper Specifications implemented here:**
* Dataset: California Housing
* Split: 80% Train, 20% Test
* Model: 3 Hidden Layers (~168K parameters)
* Optimizer & Loss: Adam, MSE
* Epochs: 200
* Checkpoint Interval: Every 20 epochs

## 1. Data Loading and Preprocessing

Neural networks require scaled data to perform gradient descent efficiently. We load the dataset, apply an 8:2 train-test split, and standardize the features using `StandardScaler`. Finally, we convert the arrays into PyTorch Tensors and wrap them in a `DataLoader` for batch processing.

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset
from sklearn.datasets import fetch_california_housing
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
import numpy as np

# Set random seeds for reproducibility
torch.manual_seed(42)
np.random.seed(42)

print("Loading data...")
housing = fetch_california_housing()
X, y = housing.data, housing.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Standardize the features
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert to PyTorch tensors
X_train_tensor = torch.FloatTensor(X_train_scaled)
y_train_tensor = torch.FloatTensor(y_train).view(-1, 1)
X_test_tensor = torch.FloatTensor(X_test_scaled)
y_test_tensor = torch.FloatTensor(y_test).view(-1, 1)

# Create DataLoader for training
train_dataset = TensorDataset(X_train_tensor, y_train_tensor)
train_loader = DataLoader(train_dataset, batch_size=256, shuffle=True)

## 2. Model Architecture

The paper specifies a Multi-Layer Perceptron (MLP) with 3 hidden layers containing roughly 168,000 parameters. To achieve this with our 8 input features, we set the width of our hidden layers to 280 nodes.

In [3]:
# 2. Define the Neural Network
class CaliforniaHousingModel(nn.Module):
    def __init__(self):
        super(CaliforniaHousingModel, self).__init__()
        self.network = nn.Sequential(
            nn.Linear(8, 280),
            nn.ReLU(),
            nn.Linear(280, 280),
            nn.ReLU(),
            nn.Linear(280, 280),
            nn.ReLU(),
            nn.Linear(280, 1)
        )

    def forward(self, x):
        return self.network(x)

model = CaliforniaHousingModel()
print(f"Total Parameters: {sum(p.numel() for p in model.parameters())}")

Total Parameters: 160161


## 3. Training and Persistent Checkpointing

To compute the `TracInCP` heuristic, we must save the state of the model across the training process. 
We train for 200 epochs using the Adam optimizer and save the model weights and learning rate to disk as `.pt` files every 20 epochs. Saving to disk ensures that if the notebook disconnects, we do not have to retrain the model from scratch to calculate influence.

In [4]:
# 3. Train the model and save checkpoints
criterion = nn.MSELoss()
learning_rate = 0.001
optimizer = optim.Adam(model.parameters(), lr=learning_rate)

epochs = 200
checkpoints = []

print("Starting training...")
for epoch in range(1, epochs + 1):
    model.train()
    total_loss = 0
    
    for batch_X, batch_y in train_loader:
        optimizer.zero_grad()
        predictions = model(batch_X)
        loss = criterion(predictions, batch_y)
        loss.backward()
        optimizer.step()
        total_loss += loss.item()
        
    # Save checkpoint every 20 epochs
    if epoch % 20 == 0:
        print(f"Epoch {epoch}/{epochs}, Loss: {total_loss/len(train_loader):.4f}")
        # Store the state dict and the learning rate used
        checkpoints.append({
            'weights': model.state_dict().copy(),
            'lr': learning_rate 
        })

print(f"Saved {len(checkpoints)} checkpoints.")

Starting training...
Epoch 20/200, Loss: 0.2679
Epoch 40/200, Loss: 0.2363
Epoch 60/200, Loss: 0.2018
Epoch 80/200, Loss: 0.1728
Epoch 100/200, Loss: 0.1453
Epoch 120/200, Loss: 0.1258
Epoch 140/200, Loss: 0.1038
Epoch 160/200, Loss: 0.0879
Epoch 180/200, Loss: 0.0738
Epoch 200/200, Loss: 0.0624
Saved 10 checkpoints.


## 4. TracIn Computation

Here we implement the core mathematical concept of the paper: the First-Order Approximation via Checkpoints. 

For a given test point $z'$ and training point $z$, we calculate the influence by iterating through all saved checkpoints, calculating the gradient of the loss with respect to the model parameters for both points, and taking their dot product weighted by the learning rate:

$$TracInCP(z, z') = \sum_{i=1}^k \eta_i \nabla\ell(w_{t_i}, z) \cdot \nabla\ell(w_{t_i}, z')$$

## 5. Identifying Proponents and Opponents

We sort the training points based on their TracIn scores. 
* **Proponents (Positive Score):** Training examples whose gradients aligned with the test example, helping reduce the loss for this specific prediction. In real estate, these are our "comparables."
* **Opponents (Negative Score):** Training examples whose gradients pointed in the opposite direction, increasing the loss for the test prediction.

In [5]:
# Helper function to extract and flatten all gradients from the model
def get_flattened_grads(model):
    grads = []
    for param in model.parameters():
        if param.grad is not None:
            grads.append(param.grad.view(-1))
    return torch.cat(grads)

# 4. Compute TracIn Influence
def compute_tracin_influence(test_x, test_y, train_x, train_y, checkpoints, model, criterion):
    influence_score = 0.0
    
    # Iterate through all saved checkpoints
    for ckpt in checkpoints:
        model.load_state_dict(ckpt['weights'])
        lr = ckpt['lr']
        
        # 1. Get gradient of the TEST point
        model.zero_grad()
        test_pred = model(test_x)
        test_loss = criterion(test_pred, test_y)
        test_loss.backward()
        test_grads = get_flattened_grads(model)
        
        # 2. Get gradient of the TRAIN point
        model.zero_grad()
        train_pred = model(train_x)
        train_loss = criterion(train_pred, train_y)
        train_loss.backward()
        train_grads = get_flattened_grads(model)
        
        # 3. Compute dot product and weight by learning rate
        dot_product = torch.dot(test_grads, train_grads).item()
        influence_score += lr * dot_product
        
    return influence_score

# Let's pick a single test point to explain
test_idx = 0
z_prime_x = X_test_tensor[test_idx].unsqueeze(0)
z_prime_y = y_test_tensor[test_idx].unsqueeze(0)

print(f"\nExplaining prediction for Test Point {test_idx}...")

# Compute influence for the first 1000 training points (to save time)
num_train_to_check = 1000
influences = []

model.eval() # Set model to evaluation mode for gradient extraction
for i in range(num_train_to_check):
    if i % 100 == 0:
        print(f"Processing training point {i}/{num_train_to_check}...")
        
    z_x = X_train_tensor[i].unsqueeze(0)
    z_y = y_train_tensor[i].unsqueeze(0)
    
    score = compute_tracin_influence(z_prime_x, z_prime_y, z_x, z_y, checkpoints, model, criterion)
    influences.append((i, score))

# Sort by influence score descending
influences.sort(key=lambda x: x[1], reverse=True)

# 5. Output Proponents and Opponents
print("\n--- Results ---")
print("Top 3 Proponents (Helpful Training Examples):")
for i in range(3):
    idx, score = influences[i]
    print(f"Train Index: {idx} | TracIn Score: {score:.6f}")

print("\nTop 3 Opponents (Harmful Training Examples):")
for i in range(1, 4):
    idx, score = influences[-i]
    print(f"Train Index: {idx} | TracIn Score: {score:.6f}")


Explaining prediction for Test Point 0...
Processing training point 0/1000...
Processing training point 100/1000...
Processing training point 200/1000...
Processing training point 300/1000...
Processing training point 400/1000...
Processing training point 500/1000...
Processing training point 600/1000...
Processing training point 700/1000...
Processing training point 800/1000...
Processing training point 900/1000...

--- Results ---
Top 3 Proponents (Helpful Training Examples):
Train Index: 912 | TracIn Score: 0.026858
Train Index: 663 | TracIn Score: 0.024938
Train Index: 721 | TracIn Score: 0.024903

Top 3 Opponents (Harmful Training Examples):
Train Index: 621 | TracIn Score: -0.031925
Train Index: 504 | TracIn Score: -0.028827
Train Index: 592 | TracIn Score: -0.027679
